# Notebook 04: XAI Audit, By-Design vs. Post-Hoc Agreement & RQ1/RQ2 Answers
## Phase 4 of the Experimental Workflow (`results/final_experimental_report.md`)

**Scientific & Methodological Objectives:**
1. **RQ1 Answer (Cross-Architectural Alignment - *XAI for Explanation*)**:
   - Evaluate whether, under the same experimental protocol, the three models (`Lopez17CNN`, `BiflowTransformer`, `MedDefenderTransformer`) converge on the same network header fields ($F=6$) and packets ($T=10$) or exhibit divergent decision strategies.
2. **RQ2 Answer (By-Design vs. Post-Hoc Audit - *Zero-Cost Faithfulness*)**:
   - **Rank Agreement (Spearman $\rho \uparrow$)**: ordering across the 10 temporal packets between SHAP and intrinsic attention (*Attention Rollout* and *MHAN*).
   - **Geometric Alignment (Cosine Similarity $CS \uparrow$)**: vector alignment in $\mathbb{R}^{10}$.
   - **Causal Field Verification (Temporal Masking)**: inference-time masking of top-$k$ packets guided by by-design attention vs. temporal SHAP to verify comparable causal faithfulness.
   - **Edge Computational Cost**: runtime gap between by-design inference (milliseconds, embedded in forward pass) and post-hoc SHAP (seconds).
   - **Table 3 Export**: generate `results/metrics/xai_audit_table.csv` for `results/final_experimental_report.md`.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve project paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
SRC_DIR = REPO_ROOT / 'src'
RESULTS_DIR = REPO_ROOT / 'results'
MODELS_DIR = RESULTS_DIR / 'models'
METRICS_DIR = RESULTS_DIR / 'metrics'
FIGURES_DIR = RESULTS_DIR / 'figures'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils import seed_everything
from util.xai_utils import compute_temporal_alignment
from xai.model_loader import load_checkpoint

seed_everything(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active device: {device}")

### 4.1 Loading XAI Explanations & Edge Latency Metrics (Task T4.1)
Load `results/metrics/xai_explanations.npz` (generated by Notebook 03) and edge latency profiles from `results/metrics/edge_latency_profiling.csv`.

In [ ]:
npz_path = METRICS_DIR / 'xai_explanations.npz'
data = np.load(npz_path, allow_pickle=True)  # trusted artifact generated by Notebook 03
shap_lopez_pkt = data['shap_lopez_pkt']
shap_biflow_pkt = data['shap_biflow_pkt']
shap_meddef_pkt = data['shap_meddef_pkt']

shap_lopez_field = data['shap_lopez_field']
shap_biflow_field = data['shap_biflow_field']
shap_meddef_field = data['shap_meddef_field']

attn_biflow_pkt = data['attn_biflow_pkt']
attn_meddef_pkt = data['attn_meddef_pkt']
field_names = list(data['field_names'])

# Explained flows from Notebook 03 (Temporal Masking on the same subset) and measured SHAP latency [ms/flow]
x_xai = torch.tensor(data['x_xai'])
y_xai = data['y_xai']
preds_xai = {'Lopez17CNN': data['pred_lopez'], 'BiflowTransformer': data['pred_biflow'],
             'MedDefenderTransformer': data['pred_meddef']}
# Real packets: semantic padding (torch.all(x == 0, dim=-1)) is excluded from alignment comparisons
valid_pkt = ~torch.all(x_xai.squeeze(1) == 0, dim=-1).numpy()  # [N, T]
shap_lat_ms = {'Lopez17CNN': float(data['shap_latency_lopez_ms']),
               'BiflowTransformer': float(data['shap_latency_biflow_ms']),
               'MedDefenderTransformer': float(data['shap_latency_meddef_ms'])}
print(f"[OK] Loaded XAI explanations from {npz_path}")

# Single-thread edge CPU latency (Table 2, generated by scripts/profile_latency.py --all)
df_lat = pd.read_csv(METRICS_DIR / 'edge_latency_profiling.csv')
if df_lat['Checkpoint'].astype(str).str.startswith('None').any():
    raise RuntimeError("edge_latency_profiling.csv was not measured on trained checkpoints: run "
                       "'python scripts/profile_latency.py --all' after the training batch job.")
lat_map = dict(zip(df_lat['Network'], df_lat['Latency_Median_ms']))

### 4.2 RQ1 Answer: Cross-Architectural Decision Strategy Convergence
Assess whether the 3 architectures focus on the same header fields ($F=6$) and temporal packet positions ($T=10$) or diverge.

In [ ]:
# Median ranking per header field
med_lopez_f = np.median(np.abs(shap_lopez_field), axis=0)
med_biflow_f = np.median(np.abs(shap_biflow_field), axis=0)
med_meddef_f = np.median(np.abs(shap_meddef_field), axis=0)

df_rq1_fields = pd.DataFrame({
    'Field': field_names,
    'Lopez17CNN_SHAP': med_lopez_f,
    'BiflowTransformer_SHAP': med_biflow_f,
    'MedDefenderTransformer_SHAP': med_meddef_f
})
df_rq1_fields['Rank_Lopez'] = df_rq1_fields['Lopez17CNN_SHAP'].rank(ascending=False).astype(int)
df_rq1_fields['Rank_Biflow'] = df_rq1_fields['BiflowTransformer_SHAP'].rank(ascending=False).astype(int)
df_rq1_fields['Rank_MedDefender'] = df_rq1_fields['MedDefenderTransformer_SHAP'].rank(ascending=False).astype(int)

# Inter-model Spearman rank correlation across header fields
rho_cnn_tr, _ = spearmanr(med_lopez_f, med_biflow_f)
rho_cnn_md, _ = spearmanr(med_lopez_f, med_meddef_f)
rho_tr_md, _ = spearmanr(med_biflow_f, med_meddef_f)

print("======================================================================")
print(" RQ1 ANSWER: CROSS-ARCHITECTURAL FEATURE ALIGNMENT")
print("======================================================================")
print(df_rq1_fields.to_string(index=False))
print(f"\nHeader Field Ranking Correlation (Spearman):")
print(f"  - Lopez17CNN vs. BiflowTransformer:      rho = {rho_cnn_tr:.3f}")
print(f"  - Lopez17CNN vs. MedDefenderTransformer:  rho = {rho_cnn_md:.3f}")
print(f"  - BiflowTransformer vs. MedDefender:      rho = {rho_tr_md:.3f}")
for col, net in [('Rank_Lopez', 'Lopez17CNN'), ('Rank_Biflow', 'BiflowTransformer'), ('Rank_MedDefender', 'MedDefenderTransformer')]:
    print(f"Dominant field for {net:24s}: {df_rq1_fields.loc[df_rq1_fields[col] == 1, 'Field'].values[0]}")

# Per-class RQ1 across all 13 classes: median |phi| profiles by Field (F=6) and Packet (T=10)
# on correctly classified flows per model, including dominant field polarity (+ red / - blue)
shap_views = {'Lopez17CNN': (shap_lopez_field, shap_lopez_pkt),
              'BiflowTransformer': (shap_biflow_field, shap_biflow_pkt),
              'MedDefenderTransformer': (shap_meddef_field, shap_meddef_pkt)}
rq1_rows = []
for c in range(13):
    row = {'Class': c}
    for net, (f_view, p_view) in shap_views.items():
        sel = (y_xai == c) & (preds_xai[net] == c)
        row[f'N_{net}'] = int(sel.sum())
        if sel.any():
            f_top = int(np.argmax(np.median(np.abs(f_view[sel]), axis=0)))
            row[f'Top_Field_{net}'] = field_names[f_top]
            # Signed median of phi~ on the dominant field: > 0 excitatory (red), < 0 inhibitory (blue)
            row[f'Signed_Phi_{net}'] = round(float(np.median(f_view[sel, f_top])), 4)
            row[f'Top_Packet_{net}'] = int(np.argmax(np.median(np.abs(p_view[sel]), axis=0))) + 1
        else:
            row[f'Top_Field_{net}'] = row[f'Signed_Phi_{net}'] = row[f'Top_Packet_{net}'] = None
    row['Field_Agreement'] = len({row[f'Top_Field_{n}'] for n in shap_views}) == 1
    row['Packet_Agreement'] = len({row[f'Top_Packet_{n}'] for n in shap_views}) == 1
    rq1_rows.append(row)
df_rq1_class = pd.DataFrame(rq1_rows)
df_rq1_class.to_csv(METRICS_DIR / 'rq1_per_class_fields.csv', index=False)
print("\nPer-class RQ1 breakdown (dominant field and packet per model):")
print(df_rq1_class.to_string(index=False))
print(f"Classes with unanimous top field across all 3 models:  {int(df_rq1_class['Field_Agreement'].sum())}/13")
print(f"Classes with unanimous top packet across all 3 models: {int(df_rq1_class['Packet_Agreement'].sum())}/13")

### 4.3 RQ2 Answer: Temporal Alignment Between Intrinsic Attention and SHAP
Compute Spearman $\rho$ and Cosine Similarity between the 10-packet SHAP temporal vector and intrinsic attention over valid packets only (semantic zero padding excluded).

In [ ]:
align_biflow = compute_temporal_alignment(shap_biflow_pkt, attn_biflow_pkt, valid_pkt)
align_meddef = compute_temporal_alignment(shap_meddef_pkt, attn_meddef_pkt, valid_pkt)

print("======================================================================")
print(" RQ2 ANSWER: INTRINSIC ATTENTION VS. GRADIENTSHAP ALIGNMENT")
print("======================================================================")
print("BiflowTransformer (Attention Rollout vs. SHAP):")
print(f"  - Spearman rho:        {align_biflow['spearman_rho_mean']:.3f} +/- {align_biflow['spearman_rho_std']:.3f}")
print(f"  - Cosine Similarity:   {align_biflow['cosine_similarity_mean']:.3f} +/- {align_biflow['cosine_similarity_std']:.3f}")
print("\nMedDefenderTransformer (MHAN Attention vs. SHAP):")
print(f"  - Spearman rho:        {align_meddef['spearman_rho_mean']:.3f} +/- {align_meddef['spearman_rho_std']:.3f}")
print(f"  - Cosine Similarity:   {align_meddef['cosine_similarity_mean']:.3f} +/- {align_meddef['cosine_similarity_std']:.3f}")

### 4.4 RQ2 Answer: Causal Verification via Temporal Masking on Frozen Models (Task T4.3)
Mask the top-$k$ packets ($k=1..5$) guided by intrinsic attention vs. temporal SHAP on frozen models (imputing the neutral value `0.5` on valid packets), evaluated on the exact subset explained in Notebook 03. Matching $\Delta\text{Macro-F1}$ drops confirm causal equivalence at zero XAI runtime cost.

In [ ]:
def mask_top_k_packets(x, ranking, k=3, mask_val=0.5):
    """
    Mask the top-k packets per flow by replacing all header fields on valid packets with mask_val.
    x: tensor [B, 1, T, F] or [B, T, F]
    ranking: array [B, T] with per-packet importance scores
    """
    x_masked = x.clone()
    for i in range(x.size(0)):
        r = ranking[i] if ranking.ndim == 2 else ranking
        top_pkts = np.argsort(-r)[:k]
        for t in top_pkts:
            if x_masked.dim() == 4:
                if x_masked[i, 0, t].abs().sum() > 0:
                    x_masked[i, 0, t, :] = mask_val
            else:
                if x_masked[i, t].abs().sum() > 0:
                    x_masked[i, t, :] = mask_val
    return x_masked

RUN_IDS = {'Lopez17CNN': 'S2_R01_Lopez17CNN_multi_10pkts_seed0',
           'BiflowTransformer': 'S2_R02_BiflowTransformer_multi_10pkts_seed0',
           'MedDefenderTransformer': 'S2_R03_MedDefender_multi_10pkts_seed0'}
eval_models = {n: load_checkpoint(exp_name=f"cic-iomt_scratch_{e}", results_dir=str(RESULTS_DIR),
                                  network=n, device=str(device))[0] for n, e in RUN_IDS.items()}

def macro_f1(m, x, bs=256):
    preds = []
    with torch.no_grad():
        for i in range(0, len(x), bs):
            out = m(x[i:i + bs].to(device))
            out = out[0] if isinstance(out, (list, tuple)) else out
            preds.append(out.argmax(dim=-1).cpu().numpy())
    return f1_score(y_xai, np.concatenate(preds), average="macro") * 100

K_STEPS = [1, 2, 3, 4, 5]
# Temporal SHAP: magnitude |phi_t|, consistent with compute_temporal_alignment.
# Padded packets are excluded from top-k selection.
only_valid = lambda r: np.where(valid_pkt, r, -np.inf)
rankings = {
    'Lopez17CNN_SHAP': ('Lopez17CNN', only_valid(np.abs(shap_lopez_pkt))),
    'BiflowTransformer_SHAP': ('BiflowTransformer', only_valid(np.abs(shap_biflow_pkt))),
    'BiflowTransformer_Attention': ('BiflowTransformer', only_valid(attn_biflow_pkt)),
    'MedDefenderTransformer_SHAP': ('MedDefenderTransformer', only_valid(np.abs(shap_meddef_pkt))),
    'MedDefenderTransformer_MHAN': ('MedDefenderTransformer', only_valid(attn_meddef_pkt)),
}
base_f1 = {n: macro_f1(m, x_xai) for n, m in eval_models.items()}
# Delta Macro-F1 [%] for k = 1..5
temporal_masking_curves = {
    key: [macro_f1(eval_models[net], mask_top_k_packets(x_xai, rank, k=k)) - base_f1[net] for k in K_STEPS]
    for key, (net, rank) in rankings.items()
}
temporal_masking_results = {key: curve[K_STEPS.index(3)] for key, curve in temporal_masking_curves.items()}  # Top-3 (Table 3)

print(f"Baseline Macro-F1 on {len(y_xai)} explained flows: " + ", ".join(f"{n} {v:.1f}%" for n, v in base_f1.items()))
for k_res, v_res in temporal_masking_results.items():
    print(f"  - {k_res:32s}: Delta F1 (Top-3) = {v_res:.1f}%")

### 4.5 Exporting Formal XAI Audit Table (`results/metrics/xai_audit_table.csv`) (Task T4.4)
Generate the comparative audit table corresponding to Section 2 of `results/final_experimental_report.md`.

In [ ]:
audit_table_records = [
    {
        'Model': 'Lopez17CNN',
        'Explanation Method': 'GradientSHAP (Post-Hoc)',
        'Spearman rho (vs SHAP)': 'N/A (Baseline)',
        'Cosine Sim (vs SHAP)': 'N/A (Baseline)',
        'Latency per Flow': f"{shap_lat_ms['Lopez17CNN'] / 1000:.2f} s",
        'Edge Computational Cost': 'Prohibitive (O(M * samples))',
        'Temporal Masking (Top-3 Delta F1)': f"{temporal_masking_results['Lopez17CNN_SHAP']:.1f}%"
    },
    {
        'Model': 'BiflowTransformer',
        'Explanation Method': 'GradientSHAP (Post-Hoc)',
        'Spearman rho (vs SHAP)': '1.000',
        'Cosine Sim (vs SHAP)': '1.000',
        'Latency per Flow': f"{shap_lat_ms['BiflowTransformer'] / 1000:.2f} s",
        'Edge Computational Cost': 'Prohibitive (O(M * samples))',
        'Temporal Masking (Top-3 Delta F1)': f"{temporal_masking_results['BiflowTransformer_SHAP']:.1f}%"
    },
    {
        'Model': 'BiflowTransformer',
        'Explanation Method': 'Attention Rollout (By-Design)',
        'Spearman rho (vs SHAP)': f"{align_biflow['spearman_rho_mean']:.3f}",
        'Cosine Sim (vs SHAP)': f"{align_biflow['cosine_similarity_mean']:.3f}",
        'Latency per Flow': f"{lat_map['BiflowTransformer']:.2f} ms",
        'Edge Computational Cost': 'Zero-cost (embedded in forward pass)',
        'Temporal Masking (Top-3 Delta F1)': f"{temporal_masking_results['BiflowTransformer_Attention']:.1f}%"
    },
    {
        'Model': 'MedDefenderTransformer',
        'Explanation Method': 'GradientSHAP (Post-Hoc)',
        'Spearman rho (vs SHAP)': '1.000',
        'Cosine Sim (vs SHAP)': '1.000',
        'Latency per Flow': f"{shap_lat_ms['MedDefenderTransformer'] / 1000:.2f} s",
        'Edge Computational Cost': 'Prohibitive (O(M * samples))',
        'Temporal Masking (Top-3 Delta F1)': f"{temporal_masking_results['MedDefenderTransformer_SHAP']:.1f}%"
    },
    {
        'Model': 'MedDefenderTransformer',
        'Explanation Method': 'MHAN Attention (By-Design)',
        'Spearman rho (vs SHAP)': f"{align_meddef['spearman_rho_mean']:.3f}",
        'Cosine Sim (vs SHAP)': f"{align_meddef['cosine_similarity_mean']:.3f}",
        'Latency per Flow': f"{lat_map['MedDefenderTransformer']:.2f} ms",
        'Edge Computational Cost': 'Zero-cost (embedded in forward pass)',
        'Temporal Masking (Top-3 Delta F1)': f"{temporal_masking_results['MedDefenderTransformer_MHAN']:.1f}%"
    }
]

df_audit_table = pd.DataFrame(audit_table_records)
out_audit_csv = METRICS_DIR / 'xai_audit_table.csv'
df_audit_table.to_csv(out_audit_csv, index=False)

print("\n======================================================================")
print(" FORMAL XAI AUDIT TABLE: TEMPORAL ALIGNMENT & FAITHFULNESS (RQ1 & RQ2)")
print("======================================================================")
display(df_audit_table) if 'display' in globals() else print(df_audit_table.to_string(index=False))
print(f"\n[SUCCESS] Audit table saved to: {out_audit_csv}")

### 4.6 Generating XAI Audit Figures with Explicit Units (Task T4.4)
Generate the 4 high-resolution (300 DPI) comparative panels with **explicit axis units**:
1. **Temporal Alignment ($t \in [1..10]$)**: normalized per-packet relevance share for SHAP vs. Attention Rollout vs. MHAN Pooling.
2. **Edge Computational Trade-off**: log-scale latency comparison [ms/flow].
3. **Causal Validation (Temporal Masking)**: performance degradation $\Delta\text{Macro-F1}$ [%] for $k=1..5$ packets.
4. **Cross-Architectural Field Convergence**: median normalized SHAP importance $|\tilde{\phi}_f|$ [a.u.] across CNN, Transformer, and MedDefender.

In [ ]:
# Publication plot styling configuration
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'figure.titlesize': 13
})

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Temporal Alignment (SHAP vs Attention Rollout vs MHAN)
ax1 = axes[0, 0]
t_steps = np.arange(1, 11)

# Per-packet shares over real packets only (padding excluded, matching compute_temporal_alignment)
share = lambda v: v.mean(axis=0) / v.mean(axis=0).sum() * 100
p_shap_biflow = share(np.abs(shap_biflow_pkt) * valid_pkt)
p_attn_biflow = share(attn_biflow_pkt)
p_shap_meddef = share(np.abs(shap_meddef_pkt) * valid_pkt)
p_attn_meddef = share(attn_meddef_pkt)
p_shap_lopez  = share(np.abs(shap_lopez_pkt) * valid_pkt)

ax1.plot(t_steps, p_shap_lopez, 'o--', color='#d95f02', label='Lopez17CNN SHAP (Post-Hoc Baseline)', linewidth=1.8)
ax1.plot(t_steps, p_shap_biflow, 's-', color='#7570b3', label='BiflowTransformer SHAP (Post-Hoc)', linewidth=2.0)
ax1.plot(t_steps, p_attn_biflow, '^--', color='#e7298a', label='BiflowTransformer Rollout (By-Design)', linewidth=2.0)
ax1.plot(t_steps, p_shap_meddef, 'D-', color='#1b9e77', label='MedDefender SHAP (Post-Hoc)', linewidth=2.0)
ax1.plot(t_steps, p_attn_meddef, 'v--', color='#66a61e', label='MedDefender MHAN (By-Design)', linewidth=2.0)

ax1.set_title('A) Temporal Agreement: SHAP vs. Intrinsic Attention (RQ1 & RQ2)', fontweight='bold')
ax1.set_xlabel('Packet Sequence Index, t [1..10]')
ax1.set_ylabel('Normalized Temporal Relevance Share [% per packet]')
ax1.set_xticks(t_steps)
ax1.set_ylim(0, 45)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper right')

# 2. Edge Computational Trade-off (Log Latency)
ax2 = axes[0, 1]
methods = ['Lopez17CNN\nGradientSHAP', 'Biflow\nGradientSHAP', 'MedDefender\nGradientSHAP', 'Biflow\nRollout (By-Design)', 'MedDefender\nMHAN (By-Design)']
latencies_ms = [shap_lat_ms['Lopez17CNN'], shap_lat_ms['BiflowTransformer'], shap_lat_ms['MedDefenderTransformer'],
                float(lat_map['BiflowTransformer']), float(lat_map['MedDefenderTransformer'])]
colors_lat = ['#e41a1c', '#e41a1c', '#e41a1c', '#377eb8', '#4daf4a']

bars = ax2.bar(methods, latencies_ms, color=colors_lat, width=0.55, edgecolor='black', linewidth=0.8)
ax2.set_yscale('log')
ax2.set_title('B) Runtime Efficiency Gap at Inference (Log Scale)', fontweight='bold')
ax2.set_xlabel('XAI Method & Architecture')
ax2.set_ylabel('Per-Flow Explanation Latency on Gateway CPU [ms]')
ax2.grid(True, which='both', linestyle='--', alpha=0.4, axis='y')
ax2.axhline(y=10.0, color='red', linestyle=':', linewidth=1.5, label='Real-Time Edge Budget (10 ms)')

for bar in bars:
    val = bar.get_height()
    txt = f'{val:.2f} ms' if val < 10 else f'{val/1000:.1f} s'
    ax2.annotate(txt, (bar.get_x() + bar.get_width() / 2, val), ha='center', va='bottom', xytext=(0, 4), textcoords='offset points', fontsize=9, fontweight='bold')
ax2.legend(loc='upper right')

# 3. Causal Validation via Temporal Masking (Top-k Packets)
ax3 = axes[1, 0]
k_steps = np.array(K_STEPS)
delta_shap_biflow = np.array(temporal_masking_curves['BiflowTransformer_SHAP'])
delta_attn_biflow = np.array(temporal_masking_curves['BiflowTransformer_Attention'])
delta_shap_meddef = np.array(temporal_masking_curves['MedDefenderTransformer_SHAP'])
delta_attn_meddef = np.array(temporal_masking_curves['MedDefenderTransformer_MHAN'])

ax3.plot(k_steps, delta_shap_biflow, 's-', color='#7570b3', linewidth=2.0, label='BiflowTransformer (SHAP-Guided)')
ax3.plot(k_steps, delta_attn_biflow, '^--', color='#e7298a', linewidth=2.0, label='BiflowTransformer (Rollout-Guided)')
ax3.plot(k_steps, delta_shap_meddef, 'D-', color='#1b9e77', linewidth=2.0, label='MedDefender (SHAP-Guided)')
ax3.plot(k_steps, delta_attn_meddef, 'v--', color='#66a61e', linewidth=2.0, label='MedDefender (MHAN-Guided)')

ax3.set_title('C) Causal Verification: Temporal Masking on Frozen Models (RQ2)', fontweight='bold')
ax3.set_xlabel('Number of Masked Top-k Packets at Inference, k [1..5]')
ax3.set_ylabel(r'Performance Drop $\Delta$ Macro-F1 [%]')
ax3.set_xticks(k_steps)
ax3.grid(True, linestyle='--', alpha=0.5)
ax3.legend(loc='lower left')

# 4. Inter-Model Convergence on the 6 Header Fields (RQ1)
ax4 = axes[1, 1]
f_indices = np.arange(len(field_names))
w = 0.25

med_lopez_f = np.median(np.abs(shap_lopez_field), axis=0)
med_biflow_f = np.median(np.abs(shap_biflow_field), axis=0)
med_meddef_f = np.median(np.abs(shap_meddef_field), axis=0)

ax4.bar(f_indices - w, med_lopez_f, width=w, color='#d95f02', label='Lopez17CNN', edgecolor='black', linewidth=0.6)
ax4.bar(f_indices, med_biflow_f, width=w, color='#7570b3', label='BiflowTransformer', edgecolor='black', linewidth=0.6)
ax4.bar(f_indices + w, med_meddef_f, width=w, color='#1b9e77', label='MedDefenderTransformer', edgecolor='black', linewidth=0.6)

ax4.set_title('D) Global Header Field Importance Across Architectures (RQ1)', fontweight='bold')
ax4.set_xlabel(r'Protocol Header Fields, $f \in \{PL, IAT, DIR, WIN, TTL, FLG\}$')
ax4.set_ylabel(r'Median Normalized SHAP Importance $|\tilde{\phi}_f|$ [a.u.]')
ax4.set_xticks(f_indices)
ax4.set_xticklabels(field_names)
ax4.grid(True, linestyle='--', alpha=0.5, axis='y')
ax4.legend(loc='upper left')

plt.tight_layout()
out_fig_audit_png = FIGURES_DIR / 'xai_audit_comprehensive_metrics.png'
out_fig_audit_pdf = FIGURES_DIR / 'xai_audit_comprehensive_metrics.pdf'
plt.savefig(out_fig_audit_png, dpi=300, bbox_inches='tight')
plt.savefig(out_fig_audit_pdf, bbox_inches='tight')
plt.show()
print(f'[SUCCESS] High-resolution XAI audit figures saved to:\n  - {out_fig_audit_png}\n  - {out_fig_audit_pdf}')